# Conditional Edge

## 조건부 Edge(`Conditional Edge`) 사용해 분기 만들기
* `Conditional Edge`를 이용해 상황에 따라 다른 노드로 연결되도록 구현할 수 있습니다.

In [1]:
from typing import Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(BaseModel):
    # Annotated[타입, 메타데이터] -> 해당 변수에 '메타데이터'에 해당하는 특성(기능)을 부여함
    # add_messages -> message에 적용하는 메타데이터, Node에서 history에 추가할 항목만 반환해도 알아서 리스트에 추가해 줌
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    last_player: Literal['A', 'B']
    number: int
    end: bool

In [8]:
import time
def node_A(state: AgentState) -> dict:
    num = state.number
    new_num = num ** 2
    time.sleep(2)
    return {
        'chat_history': [HumanMessage(name='A', content=f"숫자를 {num}에서 {new_num}(으)로 바꿈")],
        'number': new_num,
        'last_player': 'A'
    }

In [9]:
def node_B(state: AgentState) -> dict:
    num = state.number
    new_num = num - 1
    time.sleep(2)
    return {
        'chat_history': [HumanMessage(name='B', content=f"숫자를 {num}에서 {new_num}(으)로 변경")],
        'number': new_num,
        'last_player': 'B'
    }

In [10]:
def node_judge(state: AgentState) -> dict:
    end = True if state.number > 100 else False
    return {
        'chat_history': [HumanMessage(name='judge', content=f"최종 숫자 {state.number}(으)로 종료합니다" if end else "계속 진행하세요")],
        'end': end
    }

In [11]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('judge', node_judge)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'judge')
workflow.add_edge('B', 'judge')

## `route` 만들고 `conditional edge`에 부여하기
* `route`는 조건에 따라서 달라지는 목적지 `node`명 을 반환하는 함수입니다
* `workflow.add_conditional_edges`함수에 `route`를 적용하면, route가 반환한 `node`명으로 라우팅됩니다.

In [12]:
def judge_route(state: AgentState):
    # end가 True면 END로,
    if state.end:
        return END
    elif state.last_player == 'B':
        return 'A'
    else:
        return 'B'

workflow.add_conditional_edges(
    'judge', 
    judge_route
)

In [14]:
app = workflow.compile()

init_input = AgentState(chat_history=[], last_player='A', number=20, end=False)

for chunk in app.stream(init_input):
    print(chunk)

{'A': {'chat_history': [HumanMessage(content='숫자를 20에서 400(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='ed417d0d-5a8d-4b89-9d1c-622dd26e7ea3')], 'number': 400, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content='최종 숫자 400(으)로 종료합니다', additional_kwargs={}, response_metadata={}, name='judge', id='e25e932a-0b44-43bd-8b53-59d09d2a3a87')], 'end': True}}


## 토마토 게임 만들기
* ai와 토마토 게임을 진행할 수 있는 LangGraph 기반 게임을 만들어 봅시다
* 규칙: 두 명이 번갈아가면서 '토마토'를 한글자씩 말합니다. 올바르게 말하지 못하면 패배합니다.

In [16]:
from typing import Literal, Optional, Annotated

from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

TOMATO = "토마토"  # len=3, turn % 3 으로 순환

class TomatoGameState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    next_player: Literal['user', 'ai'] = 'user'
    turn: int = 0
    game_over: bool = False
    winner: Optional[Literal['user', 'ai']] = None
    player_input: Optional[str] = None

In [23]:
import time

def user_node(state: TomatoGameState) -> dict:
    expected = TOMATO[state.turn % len(TOMATO)]
    user_input = input(f"다음 글자를 말하세요: ").strip()
    return {
        'player_input': user_input,
        'chat_history': [HumanMessage(name='user', content=user_input)],
    }

def ai_node(state: TomatoGameState) -> dict:
    expected = TOMATO[state.turn % len(TOMATO)]
    time.sleep(1)
    print(f"AI: {expected}")
    return {
        'player_input': expected,
        'chat_history': [HumanMessage(name='ai', content=expected)],
    }

def judge_node(state: TomatoGameState) -> dict:
    expected = TOMATO[state.turn % len(TOMATO)]
    current = state.next_player  # 지금 말한 사람

    if state.player_input != expected:
        winner = 'ai' if current == 'user' else 'user'
        return {
            'game_over': True,
            'winner': winner,
            'chat_history': [
                HumanMessage(
                    name='judge',
                    content=f"'{state.player_input}' 틀림! 정답은 '{expected}'. {winner} 승리!",
                )
            ],
        }

    next_player = 'ai' if current == 'user' else 'user'
    return {
        'turn': state.turn + 1,
        'next_player': next_player,
        'chat_history': [
            HumanMessage(name='judge', content=f"정답! 다음은 {next_player}")
        ],
    }

In [24]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(TomatoGameState)

workflow.add_node('user', user_node)
workflow.add_node('ai', ai_node)
workflow.add_node('judge', judge_node)

def start_route(state: TomatoGameState):
    return 'user' if state.next_player == 'user' else 'ai'

def judge_route(state: TomatoGameState):
    if state.game_over:
        return END
    return 'user' if state.next_player == 'user' else 'ai'

workflow.add_conditional_edges(START, start_route)
workflow.add_edge('user', 'judge')
workflow.add_edge('ai', 'judge')
workflow.add_conditional_edges('judge', judge_route)

app = workflow.compile()

In [22]:
init = TomatoGameState()  # user가 먼저 시작

for chunk in app.stream(init):
    print(chunk)

{'user': {'player_input': '토', 'chat_history': [HumanMessage(content='토', additional_kwargs={}, response_metadata={}, name='user', id='91976832-b8c9-405f-be4e-25e93706bc34')]}}
{'judge': {'turn': 1, 'next_player': 'ai', 'chat_history': [HumanMessage(content='정답! 다음은 ai', additional_kwargs={}, response_metadata={}, name='judge', id='9780e4ef-4ebf-4b6e-a9a6-31a0b22c750a')]}}
AI: 마
{'ai': {'player_input': '마', 'chat_history': [HumanMessage(content='마', additional_kwargs={}, response_metadata={}, name='ai', id='376ca2c0-3284-4518-b5de-9c222584595b')]}}
{'judge': {'turn': 2, 'next_player': 'user', 'chat_history': [HumanMessage(content='정답! 다음은 user', additional_kwargs={}, response_metadata={}, name='judge', id='7cba226d-4511-4ff2-aa5d-124868e1b5e5')]}}
{'user': {'player_input': '토', 'chat_history': [HumanMessage(content='토', additional_kwargs={}, response_metadata={}, name='user', id='619ad550-6e8c-4df9-95eb-7bfd33d8f947')]}}
{'judge': {'turn': 3, 'next_player': 'ai', 'chat_history': [Human

### 팬아웃(Fan-Out) 구현하기
* 하나의 노드에서 여러 노드로 갈라지고(팬아웃) 다시 한 노드로 흐름을 모을 (팬인) 수 있습니다

In [25]:
from pydantic import BaseModel
from typing import Optional, Any

class FanOutState(BaseModel):
    value1: Optional[Any] = None
    value2: Optional[Any] = None

In [ ]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
        state.value1 = state.value1 + 1
        print(f"value1={state.value1}")
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
        print(f"value2={state.value2}")
    return {"value2": state.value2 + 1}

In [49]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')

app = workflow.compile()

In [50]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
value1=1
B 노드 실행 중... (2/3)
value2=0
C 노드 실행 중... (2/3)
value2=0value1=2
B 노드 실행 중... (3/3)

C 노드 실행 중... (3/3)
value2=0value1=3



{'value1': 4, 'value2': 1}

## 팬인(Fan-In) 구현하기
* `add_edge` 함수의 출발점은 여러 노드의 배열도 입력받을 수 있습니다.

In [30]:
def node_D(state: FanOutState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}

In [31]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [32]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)
C 노드 실행 중... (2/3)
C 노드 실행 중... (3/3)B 노드 실행 중... (3/3)

D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

### Q. 팬인 시 노드 종료 시점이 다르다면?

In [40]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 5):
        print(f"C 노드 실행 중... ({i}/4)")
        time.sleep(5)
    return {"value2": state.value2 + 1}

In [41]:
def node_D(state: FanOutState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}

In [42]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [43]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/4)
B 노드 실행 중... (2/3)
B 노드 실행 중... (3/3)
C 노드 실행 중... (2/4)
C 노드 실행 중... (3/4)
C 노드 실행 중... (4/4)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

# `Reducer` 활용하기

In [51]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

    

In [52]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [53]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
C 노드 실행 중... (2/3)
B 노드 실행 중... (2/5)
C 노드 실행 중... (3/3)B 노드 실행 중... (3/5)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


InvalidUpdateError: At key 'value1': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

In [54]:
from typing import Annotated

def numberAddReducer(left: int, right: int) -> int: # left: 기존 State, right: return 받은 state
    return left + right # return 받은 값으로 대체하지 않고, 기존 값에 return 받은 값을 더함

class ReducerState(BaseModel):
    value1: Annotated[Optional[Any], numberAddReducer] = None
    value2: Optional[Any] = None

In [55]:
def node_B(state: ReducerState) -> dict: # state 정의도 모두 변경
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": 1} # 새로 더해줄 값만 반환하면 됨

def node_C(state: ReducerState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": 1}

def node_D(state: ReducerState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": 1}


In [56]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [57]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
C 노드 실행 중... (2/3)
B 노드 실행 중... (2/5)
B 노드 실행 중... (3/5)C 노드 실행 중... (3/3)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 2, 'value2': 1}

### 동적 팬아웃 구현하기
* `Conditional Edge`를 이용하면 Fan-Out을 동적으로 설정할 수 있습니다
* Fan-Out을 설정하려면, 목적지 노드 이름을 리스트로 모두 전달하면 됩니다.

In [58]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')

def fanout_router(state: ReducerState):
    if state.value2 > 0:
        return 'B'
    else:
        return ['B', 'C']
    
workflow.add_conditional_edges('A', fanout_router)

workflow.add_edge(['B', 'C'], 'D')

def end_router(state: ReducerState):
    if state.value1 > 2:
        return END
    else:
        return 'A'

workflow.add_conditional_edges('D', end_router)


app = workflow.compile()

In [59]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)
C 노드 실행 중... (2/3)
B 노드 실행 중... (3/5)
C 노드 실행 중... (3/3)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)
A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
B 노드 실행 중... (2/5)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


{'value1': 3, 'value2': 1}

### 라우팅 심화 문제
* Fan-Out과 Reducing, 조건부 Edge를 적극적으로 활용해 음식점 운영을 최적화해봅시다 (괄호 안의 숫자는 소요시간(초))
```
burger: patty(5), bun(3) 각각 필요 -> burgerSet(2)
fries: fry(3) -> friesSet(1)
beverage: beverage(2)

세 가지가 모두 준비되면 종료
```

In [61]:
from pydantic import BaseModel
from typing import List, Annotated, Literal, Optional

def menuReducer(left:List[str], right:List[str]) -> List[str]:
    return left + right

class RestaurantState(BaseModel):
    menu_ordered: List[Literal['burger', 'fries', 'beverage']]
    menu_complete: Annotated[Optional[List[Literal['burger', 'fries', 'beverage']]], menuReducer] = []


In [62]:
import time

# 각종 그래프 노드 실행 및 상태 갱신 통합: 반환값으로 state의 menu_complete를 적절히 업데이트하도록 구성
def g_burger_start(state: RestaurantState) -> dict:
    print("버거 준비 시작")
    return {}

def g_fries_start(state: RestaurantState) -> dict:
    print("감자튀김 준비 시작")
    return {}

def g_patty(state: RestaurantState) -> dict:
    print("패티 조리 시작")
    time.sleep(5)
    print("패티 조리 완료")
    return {}

def g_bun(state: RestaurantState) -> dict:
    print("번 조리 시작")
    time.sleep(3)
    print("번 조리 완료")
    return {}

def g_burger_set(state: RestaurantState) -> dict:
    print("버거 세트 조립 시작")
    time.sleep(2)
    print("버거 세트 조립 완료")
    return {}

def g_burger_complete(state: RestaurantState) -> dict:
    print("버거 완성")
    return {'menu_complete': ['burger']}

def g_fry(state: RestaurantState) -> dict:
    print("감자튀김 조리 시작")
    time.sleep(3)
    print("감자튀김 조리 완료")
    return {}

def g_fries_set(state: RestaurantState) -> dict:
    print("감자튀김 세트 준비 시작")
    time.sleep(1)
    print("감자튀김 세트 준비 완료")
    return {}

def g_fries_complete(state: RestaurantState) -> dict:
    print("감자튀김 완성")
    return {'menu_complete': ['fries']}

def g_beverage(state: RestaurantState) -> dict:
    print("음료 준비 시작")
    time.sleep(2)
    print("음료 준비 완료")
    return {'menu_complete': ['beverage']}

In [68]:
from langgraph.graph import StateGraph, START, END

def g_check_done(state: RestaurantState) -> dict:
  ordered = set(state.menu_ordered)
  complete = set(state.menu_complete or [])
  if ordered <= complete:
    print(f"모든 주문 완료: {sorted(complete)}")
  return {}

def g_sink(state: RestaurantState) -> dict:
  return {}  # 이 갈래만 종료, 다른 병렬 작업은 계속 진행

workflow = StateGraph(RestaurantState)

# 노드 등록 (기존 g_* 함수 그대로 사용)
workflow.add_node('burger_start', g_burger_start)
workflow.add_node('fries_start', g_fries_start)
workflow.add_node('patty', g_patty)
workflow.add_node('bun', g_bun)
workflow.add_node('burger_set', g_burger_set)
workflow.add_node('burger_complete', g_burger_complete)
workflow.add_node('fry', g_fry)
workflow.add_node('fries_set', g_fries_set)
workflow.add_node('fries_complete', g_fries_complete)
workflow.add_node('beverage', g_beverage)
workflow.add_node('check_done', g_check_done)
workflow.add_node('sink', g_sink)

# 1) START → 주문된 메뉴만 동적 Fan-Out
def dispatch_route(state: RestaurantState):
    targets = []
    if 'burger' in state.menu_ordered:
        targets.append('burger_start')
    if 'fries' in state.menu_ordered:
        targets.append('fries_start')
    if 'beverage' in state.menu_ordered:
        targets.append('beverage')
    if not targets:
        return END
    return targets[0] if len(targets) == 1 else targets

workflow.add_conditional_edges(START, dispatch_route)

# 2) 버거: patty + bun 병렬 → Fan-In → 세트 조립
workflow.add_edge('burger_start', 'patty')
workflow.add_edge('burger_start', 'bun')
workflow.add_edge(['patty', 'bun'], 'burger_set')
workflow.add_edge('burger_set', 'burger_complete')

# 3) 감자튀김: fry → fries_set (순차)
workflow.add_edge('fries_start', 'fry')
workflow.add_edge('fry', 'fries_set')
workflow.add_edge('fries_set', 'fries_complete')

# 4) 완료 노드 → check_done
workflow.add_edge('burger_complete', 'check_done')
workflow.add_edge('fries_complete', 'check_done')
workflow.add_edge('beverage', 'check_done')

# 5) 주문 전부 완료됐을 때만 END
def check_done_route(state: RestaurantState):
    if set(state.menu_ordered) <= set(state.menu_complete or []):
        return END
    return 'sink'  # 먼저 끝난 갈래는 여기서 종료

workflow.add_conditional_edges('check_done', check_done_route)
workflow.add_edge('sink', END)

app = workflow.compile()

In [64]:
def get_menu_ordered() -> list[str]:
    print("주문할 메뉴를 입력하세요 (쉼표로 구분)")
    print("  예: burger")
    print("  예: burger,fries")
    print("  예: burger,fries,beverage")
    raw = input("주문 > ").strip().lower()

    valid = {'burger', 'fries', 'beverage'}
    ordered = [item.strip() for item in raw.split(',') if item.strip()]

    # 유효성 검사
    invalid = [x for x in ordered if x not in valid]
    if invalid:
        raise ValueError(f"잘못된 메뉴: {invalid}. 가능한 값: {sorted(valid)}")

    if not ordered:
        raise ValueError("주문이 비어 있습니다.")

  # 중복 제거 (burger,burger → burger)
    return list(dict.fromkeys(ordered))

In [69]:
menu_ordered = get_menu_ordered()
print(f"주문 접수: {menu_ordered}\n")

init_state = RestaurantState(menu_ordered=menu_ordered)
result = app.invoke(init_state)

print("\n최종 state:")
print(result)

주문할 메뉴를 입력하세요 (쉼표로 구분)
  예: burger
  예: burger,fries
  예: burger,fries,beverage
주문 접수: ['beverage', 'burger']

음료 준비 시작
버거 준비 시작
음료 준비 완료
번 조리 시작
패티 조리 시작
번 조리 완료
패티 조리 완료
버거 세트 조립 시작
버거 세트 조립 완료
버거 완성
모든 주문 완료: ['beverage', 'burger']

최종 state:
{'menu_ordered': ['beverage', 'burger'], 'menu_complete': ['beverage', 'burger']}
